# Tutorial 1 — Explore the LeMat-Synth dataset

**LeMat-Synth** is the published output of this toolbox: ~58k synthesis
procedures extracted from open-access materials science papers and structured
into the `GeneralSynthesisOntology` schema, each one scored by an LLM judge.

This tutorial is the cheapest way into the project — you read the *results* of
the pipeline instead of running it, so **no LLM API keys and no GPU are
needed**.

## What you'll learn

1. How to load the dataset and pick the right config for your use case
2. What every column means, and how the nested `structured_synthesis` record
   maps onto the Pydantic ontology in this repo
3. How to slice the dataset by synthesis method, material category and
   **judge score**
4. How to pull one recipe apart step by step, and validate it against the
   library's own schema
5. How to export a filtered subset for your own downstream work

## Prerequisites

- The package installed (`uv sync && uv pip install -e .`) — see the
  [Installation guide](https://lematerial.github.io/lematerial-llm-synthesis/getting-started/installation/)
- Access to the gated dataset (Step 0 below)
- **Runtime:** a few minutes. **Cost:** free — no LLM calls at all.

## Step 0 — Dataset access

[`LeMaterial/LeMat-Synth`](https://huggingface.co/datasets/LeMaterial/LeMat-Synth)
is a **gated** dataset: you have to request access once on the Hub, then
authenticate locally. Two ways to authenticate — either works:

**Option A (recommended) — log in once, globally:**

```bash
hf auth login          # paste a token from huggingface.co/settings/tokens
```

**Option B — put a token in `.env`,** the same file this project uses for LLM
keys. From the repository root:

```bash
cp .env.example .env
```

then add a line to `.env`:

```
HF_TOKEN=hf_...
```

`.env` is git-ignored, so the token never gets committed. The `huggingface_hub`
library picks `HF_TOKEN` up from the environment automatically once `.env` is
loaded — which is what the next cell does.

In [ ]:
import os

from dotenv import find_dotenv, load_dotenv

# find_dotenv walks up from the working directory, so this works whether you
# started Jupyter at the repo root or inside this folder.
env_path = find_dotenv(usecwd=True)
load_dotenv(env_path, override=True)

print(
    f".env loaded from: {env_path or 'NOT FOUND (that is fine if you ran hf auth login)'}"
)

if os.getenv("HF_TOKEN"):
    print(
        f"HF_TOKEN found in the environment ({len(os.getenv('HF_TOKEN'))} chars)"
    )
else:
    print(
        "No HF_TOKEN in the environment - relying on `hf auth login` instead."
    )

# Confirm we can actually authenticate before downloading anything.
from huggingface_hub import whoami

try:
    print(f"Authenticated as: {whoami()['name']}")
except Exception as exc:
    raise RuntimeError(
        "Not authenticated with the HuggingFace Hub. Run `hf auth login`, or "
        "add HF_TOKEN=... to your .env at the repository root."
    ) from exc

## Step 1 — Choose a config and load it

The dataset ships in three configs, each split by paper source
(`arxiv`, `chemrxiv`, `omg24`):

| Config | What it is | Size |
|--------|-----------|------|
| `full` | Every extraction, whatever the judge thought of it | ~58k rows |
| `high_score` | Only extractions the judge rated highly — the one to use if you want clean training data | ~30k rows |
| `NeurIPS-AI4Mat-2025` | The frozen subset behind the paper, with extra columns | ~2.8k rows |

We start with `full/omg24` because it is the smallest of the `full` splits
(~61 MB) — swap in `arxiv` (~444 MB) once you know what you want.

> **Tip:** the first call downloads and caches parquet files under
> `~/.cache/huggingface`. Re-running the cell afterwards is instant.

In [ ]:
from datasets import load_dataset

CONFIG = "full"  # "full" | "high_score" | "NeurIPS-AI4Mat-2025"
SPLIT = "omg24"  # "arxiv" | "chemrxiv" | "omg24"

ds = load_dataset("LeMaterial/LeMat-Synth", CONFIG, split=SPLIT)

print(f"{CONFIG}/{SPLIT}: {len(ds):,} extracted synthesis procedures")
print(f"\nColumns: {ds.column_names}")

### If the download is too big

Every config can be streamed instead of downloaded, which is handy for the
444 MB `arxiv` split when you only want to peek:

```python
stream = load_dataset(
    "LeMaterial/LeMat-Synth", "full", split="arxiv", streaming=True
)
first = next(iter(stream))
```

Streaming datasets don't support `len()` or random indexing, so the rest of
this tutorial assumes the downloaded version.

## Step 2 — What is in a row?

Each row is **one material from one paper** — a paper that synthesised five
compounds contributes five rows.

| Column | Type | Meaning |
|--------|------|---------|
| `synthesized_material` | str | The material this row describes |
| `material_category` | str | One of the 16 `target_compound_type` values |
| `synthesis_method` | str | One of the 35 `synthesis_method` values |
| `structured_synthesis` | struct | The full recipe: precursors, ordered steps, conditions, equipment |
| `evaluation` | struct | The LLM judge's scores and reasoning for this extraction |
| `paper_title`, `paper_doi`, `paper_url`, `paper_abstract`, `paper_published_date` | str | Provenance — always cite the source paper |

`material_category` and `synthesis_method` are *closed* vocabularies. They come
from the `Literal` enums on `GeneralSynthesisOntology`
(`src/llm_synthesis/models/ontologies/general.py`), which is the same schema the
extractor writes into and the judge reads from.

In [ ]:
row = ds[42]

print(f"Material : {row['synthesized_material']}")
print(f"Category : {row['material_category']}")
print(f"Method   : {row['synthesis_method']}")
print(f"Paper    : {row['paper_title'][:80]}")
print(f"URL      : {row['paper_url']}")

print("\nstructured_synthesis keys:")
for key, value in row["structured_synthesis"].items():
    kind = type(value).__name__
    size = f" ({len(value)} items)" if isinstance(value, list) else ""
    print(f"  {key:<22} {kind}{size}")

print("\nevaluation keys:", list(row["evaluation"]))

## Step 3 — Slice by method and category

Because the vocabularies are closed, plain value counts are meaningful — no
string normalisation needed.

In [ ]:
import pandas as pd

methods = pd.Series(ds["synthesis_method"]).value_counts()
categories = pd.Series(ds["material_category"]).value_counts()

print("Top 10 synthesis methods")
print(methods.head(10).to_string())
print("\nTop 10 material categories")
print(categories.head(10).to_string())

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

methods.head(12)[::-1].plot.barh(ax=axes[0], color="#4C72B0")
axes[0].set_title(f"Synthesis methods - {CONFIG}/{SPLIT}")
axes[0].set_xlabel("procedures")

categories.head(12)[::-1].plot.barh(ax=axes[1], color="#55A868")
axes[1].set_title(f"Material categories - {CONFIG}/{SPLIT}")
axes[1].set_xlabel("procedures")

plt.tight_layout()
plt.show()

In [ ]:
# Filter to a single method. HuggingFace `filter` is lazy-friendly and keeps
# the result a Dataset, so you can keep chaining.
METHOD = "hydrothermal"

subset = ds.filter(lambda row: row["synthesis_method"] == METHOD)
print(f"{len(subset):,} {METHOD} procedures")

for row in subset.select(range(min(5, len(subset)))):
    print(
        f'  - {row["synthesized_material"]:<28} Paper: "{row["paper_title"][:60]}"'
    )

## Step 4 — Filter by extraction quality

Every extraction was scored 1–5 by an LLM judge across seven dimensions
(structural completeness, material extraction, process steps, equipment,
conditions, semantic accuracy, format compliance) plus an `overall_score`.
The same judge lives in this repo as `DspyGeneralSynthesisJudge` — Tutorial 6
shows how to run it yourself.

Scores are the honest way to trade recall for precision: `full` gives you
everything, a score threshold gives you the subset you can trust. (The
`high_score` config is a pre-filtered version of exactly this idea.)

In [ ]:
scores = pd.DataFrame(
    {
        "overall": [e["scores"]["overall_score"] for e in ds["evaluation"]],
        "steps": [e["scores"]["process_steps_score"] for e in ds["evaluation"]],
        "conditions": [
            e["scores"]["conditions_extraction_score"] for e in ds["evaluation"]
        ],
        "semantic": [
            e["scores"]["semantic_accuracy_score"] for e in ds["evaluation"]
        ],
    }
)

print(scores.describe().round(2).to_string())

fig, ax = plt.subplots(figsize=(7, 4))
scores["overall"].plot.hist(bins=20, ax=ax, color="#C44E52", edgecolor="white")
ax.set_xlabel("judge overall_score")
ax.set_title(f"Extraction quality - {CONFIG}/{SPLIT}")
plt.tight_layout()
plt.show()

In [ ]:
SCORE_THRESHOLD = 4.0

high_quality = ds.filter(
    lambda row: (
        (row["evaluation"]["scores"]["overall_score"] or 0) >= SCORE_THRESHOLD
    )
)

kept = len(high_quality) / len(ds) * 100
print(
    f"overall_score >= {SCORE_THRESHOLD}: "
    f"{len(high_quality):,} / {len(ds):,} rows ({kept:.1f}%)"
)

## Step 5 — Read one recipe properly

`structured_synthesis` is the same shape as `GeneralSynthesisOntology`, so you
can load it straight back into the Pydantic model that the rest of this library
uses. That gives you attribute access, type coercion and validation for free
and it is the bridge between "dataset row" and "object my code can work with".

This could be useful for example if you want to do your own downstream
processing, or if you want to validate the dataset against the schema to check
for errors.

In [ ]:
from llm_synthesis.models.ontologies.general import GeneralSynthesisOntology

example = high_quality[0] if len(high_quality) else ds[42]
recipe = GeneralSynthesisOntology.model_validate(
    example["structured_synthesis"]
)

print(f"Target compound : {recipe.target_compound}")
print(f"Compound type   : {recipe.target_compound_type}")
print(f"Method          : {recipe.synthesis_method}")

print("\nStarting materials:")
for mat in recipe.starting_materials:
    amount = f"{mat.amount} {mat.unit}" if mat.amount is not None else "n/a"
    print(f"  - {mat.name:<32} {amount}")

print("\nSteps:")
for step in recipe.steps:
    print(f"  {step.step_number}. {step.action}")
    if step.conditions:
        c = step.conditions
        bits = []
        if c.temperature is not None:
            bits.append(f"{c.temperature} {c.temp_unit or ''}".strip())
        if c.duration is not None:
            bits.append(f"{c.duration} {c.time_unit or ''}".strip())
        if c.atmosphere:
            bits.append(c.atmosphere)
        if bits:
            print(f"       conditions: {', '.join(bits)}")

print(
    "\nEquipment:",
    ", ".join(e.name for e in recipe.equipment) or "none recorded",
)
print(f"\nSource: {example['paper_title']}\n        {example['paper_url']}")

> **Note on `model_validate`.** If a row ever fails validation, that is
> informative rather than fatal: it means the published record predates a schema
> change. Wrap the call in `try/except pydantic.ValidationError` when you sweep
> the whole dataset, and count the failures.

## Step 6 — Export a working subset

A flat table is usually what you want downstream (pandas, a CSV for
collaborators, a training set). Keep the provenance columns — every row is a
claim about somebody's paper.

In [ ]:
from pathlib import Path


def repo_root(start: Path | None = None) -> Path:
    """Walk up from `start` (default: cwd) until a directory has pyproject.toml."""
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError(f"No pyproject.toml found above {here}")


# data/ is git-ignored, so exports never end up in a commit by accident.
OUTPUT_DIR = repo_root() / "data" / "tutorials"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

records = []
for row in high_quality:
    synthesis = row["structured_synthesis"]
    records.append(
        {
            "material": row["synthesized_material"],
            "category": row["material_category"],
            "method": row["synthesis_method"],
            "n_steps": len(synthesis["steps"]),
            "n_precursors": len(synthesis["starting_materials"]),
            "overall_score": row["evaluation"]["scores"]["overall_score"],
            "paper_title": row["paper_title"],
            "paper_doi": row["paper_doi"],
            "paper_url": row["paper_url"],
        }
    )

df = pd.DataFrame(records)
csv_path = (
    OUTPUT_DIR / f"lemat_synth_{CONFIG}_{SPLIT}_score{SCORE_THRESHOLD}.csv"
)
df.to_csv(csv_path, index=False)

print(f"Wrote {len(df):,} rows to {csv_path}")
df.head(10)

## What's next

- **[Tutorial 2 — Finding papers](02_finding_papers.ipynb)**: search the raw
  paper corpus that feeds this dataset, so you can build your own subset.
- **[Tutorial 3 — Synthesis + performance from a paper](03_extracting_synthesis_and_performance.ipynb)**:
  run the extraction pipeline on a paper of your own and produce rows exactly
  like the ones you just read.
- **[Tutorial 5 — Evaluating extraction quality](05_evaluating_extraction_quality.ipynb)**:
  run the same judge that produced the `evaluation` column, and compare it to
  human annotations.

If you use the dataset in published work, cite
[LeMat-Synth v1.0](https://arxiv.org/abs/2510.26824) (NeurIPS AI4Mat 2025) and
the source papers behind the rows you used.